In [ ]:
import os
import json
import shutil

DATASETS_CONFIG = {
    "Stanford_Car_Modified": {
        "json_path": "/Users/chutiangong/Downloads/Stanford_Car.v8-classes-modified_raw-images.coco/valid/_annotations.coco.json",
        "image_dir": "/Users/chutiangong/Downloads/Stanford_Car.v8-classes-modified_raw-images.coco/valid",
        "output_dir": "/Users/chutiangong/Downloads/Stanford_Car.v8-classes-modified_raw-images.coco/valid/cleaned_dataset",
        "target_category_ids": [2, 3, 4, 7]
    }
}# Manually change the above paths and category IDs for datasets

MIN_FILE_SIZE_KB = 30
MIN_AREA_RATIO = 0.20
MAX_AREA_RATIO = 0.50
MIN_SINGLE_MARGIN = 0.05
MIN_SUM_MARGIN = 0.3

for dataset_name, config in DATASETS_CONFIG.items():
    json_path = config["json_path"]
    image_dir = config["image_dir"]
    output_dir = config["output_dir"]
    target_ids = config["target_category_ids"]
    
    if not json_path or not os.path.exists(json_path):
        print(f"Skipping {dataset_name}: json_path not found.")
        continue
        
    os.makedirs(output_dir, exist_ok=True)
    
    with open(json_path, 'r', encoding='utf-8') as f:
        coco_data = json.load(f)

    images_info = {img['id']: img for img in coco_data['images']}
    annotations_map = {}
    filtered_out_by_class = 0
    
    for ann in coco_data['annotations']:
        cat_id = ann.get('category_id')
        if target_ids and (cat_id not in target_ids):
            filtered_out_by_class += 1
            continue
            
        img_id = ann['image_id']
        if img_id not in annotations_map:
            annotations_map[img_id] = []
        annotations_map[img_id].append(ann)

    valid_records = []
    total_images = len(images_info)
    discard_size = 0
    discard_ratio = 0
    discard_no_valid_box = 0
    discard_margin_single = 0
    discard_margin_sum = 0
    filtered_out_by_aspect_ratio = 0

    for img_id, img_data in images_info.items():
        file_name = img_data['file_name']
        img_path = os.path.join(image_dir, file_name)
        
        if not os.path.exists(img_path):
            continue
        
        if os.path.getsize(img_path) / 1024.0 < MIN_FILE_SIZE_KB:
            discard_size += 1
            continue
            
        if img_id not in annotations_map:
            discard_no_valid_box += 1
            continue
            
        img_width = img_data['width']
        img_height = img_data['height']
        total_area = img_width * img_height
        
        max_box_area = 0
        best_bbox = None
        best_cat_id = None
        
        for ann in annotations_map[img_id]:
            bbox = ann.get('bbox', [])
            if len(bbox) != 4:
                continue
            box_w, box_h = bbox[2], bbox[3]
            
            if box_w <= box_h:
                filtered_out_by_aspect_ratio += 1
                continue
                
            area = box_w * box_h
            if area > max_box_area:
                max_box_area = area
                best_bbox = bbox
                best_cat_id = ann.get('category_id')
                
        if max_box_area == 0:
            discard_no_valid_box += 1
            continue
            
        area_ratio = max_box_area / total_area
        if not (MIN_AREA_RATIO <= area_ratio <= MAX_AREA_RATIO):
            discard_ratio += 1
            continue
            
        x_min, y_min, box_w, box_h = best_bbox
        x_max = x_min + box_w
        y_max = y_min + box_h
        
        margin_left = x_min / img_width
        margin_right = (img_width - x_max) / img_width
        margin_top = y_min / img_height
        margin_bottom = (img_height - y_max) / img_height
        
        if min(margin_left, margin_right, margin_top, margin_bottom) < MIN_SINGLE_MARGIN:
            discard_margin_single += 1
            continue
            
        sum_margin_x = margin_left + margin_right  
        sum_margin_y = margin_top + margin_bottom  
        
        margin_x_pass = sum_margin_x >= MIN_SUM_MARGIN
        margin_y_pass = sum_margin_y >= MIN_SUM_MARGIN
        
        if not (margin_x_pass or margin_y_pass):
            discard_margin_sum += 1
            continue

        valid_records.append({
            "file_name": file_name,
            "bbox": best_bbox,
            "category_id": best_cat_id,
            "ratio": round(area_ratio, 4)
        })
        shutil.copy(img_path, os.path.join(output_dir, file_name))

    with open(os.path.join(output_dir, "cleaned_index.json"), 'w', encoding='utf-8') as f:
        json.dump(valid_records, f, indent=4)

    print(f"Dataset: {dataset_name}")
    print(f"Total input images: {total_images}")
    print(f"Discarded (category): {filtered_out_by_class}")
    print(f"Discarded (aspect ratio): {filtered_out_by_aspect_ratio}")
    print(f"Discarded (file size): {discard_size}")
    print(f"Discarded (no valid box): {discard_no_valid_box}")
    print(f"Discarded (area ratio): {discard_ratio}")
    print(f"Discarded (single margin): {discard_margin_single}")
    print(f"Discarded (sum margin): {discard_margin_sum}")
    print(f"Retained images: {len(valid_records)}")
    print(f"Retention rate: {len(valid_records) / total_images * 100:.2f}%\n")